<a href="https://colab.research.google.com/github/s-zeidi/BIS/blob/main/BIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install pm4py
The installation and import of pm4py are crucial for our Business Information
Systems (BIS) project.
pm4py enables comprehensive analysis of business processes, allowing us to
identify inefficiencies,
optimize resource allocation, and ensure regulatory compliance and risk
management through meticulous
examination of event logs.

# Initial settings

In [ ]:
!pip install pm4py
!pip install wordcloud
!pip3 install xes_exporter


In [ ]:
import os

# Check if CUDA_HOME is set, if not, set it to the appropriate CUDA installation directory
if 'CUDA_HOME' not in os.environ:
    os.environ['CUDA_HOME'] = '/usr/local/cuda'

# Set LD_LIBRARY_PATH to include CUDA libraries
if 'LD_LIBRARY_PATH' not in os.environ:
    os.environ['LD_LIBRARY_PATH'] = os.path.join(os.environ['CUDA_HOME'], 'lib64')
try:
    import pm4py
    # Your code that depends on pm4py
except ImportError as e:
    print("Error importing pm4py:", e)


In [ ]:
import pm4py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator
from scipy.stats import chi2_contingency
from wordcloud import WordCloud

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
file_path='/content/drive/MyDrive/BIS/'

# Importing .csv and .xes








In this section of code, we define two different functions for importing CSV and XES files.

In [ ]:
def import_csv(file_path):
    event_log= pd.read_csv(file_path, sep=';')
    num_event = len(event_log)
    num_case= len(event_log.case_id.unique())
    #print(event_log, "\nnum event=" + str(num_event), "\nnum case=" + str(num_case))
    event_log=pm4py.format_dataframe(event_log,case_id='case_id',activity_key='activity',timestamp_key='timestamp',timest_format='%Y-%m-%d %H:%M:%S%z')
    start_activity=pm4py.get_start_activities(event_log)
    end_activity=pm4py.get_end_activities(event_log)
    print("statr_activity=" + "{}".format(start_activity), "end_activity=" + "{}".format(end_activity))
    return event_log

def import_xes(file_path):
  event_log=pm4py.read_xes(file_path)
  start_activity=pm4py.get_start_activities(event_log)
  end_activity=pm4py.get_end_activities(event_log)
  #print(event_log)
  print("statr_activity=" + "{}".format(start_activity), "end_activity=" + "{}".format(end_activity))
  return event_log



In [ ]:
file_path = '/content/drive/MyDrive/BIS/running-example.csv'
file_path1 = '/content/drive/MyDrive/BIS/BPI_Challenge_2019.xes'



In [ ]:
#log=import_csv('/content/drive/MyDrive/BIS/running-example.csv')
log=import_xes("/content/drive/MyDrive/BIS/BPI_Challenge_2019.xes")

In [ ]:
log.head()

In [ ]:
column_names = log.columns
print(column_names)

#1.Preprocessing

##1.1.Exploring Log File





This code segment explores a log file by analyzing the frequency of each unique sequence of events, known as variants. It generates a sorted list of variants based on their frequencies, with the most frequent variants appearing first.

In [ ]:
variants=pm4py.get_variants(log)
variants = sorted(variants.items(), key=lambda x: x[1],reverse=True)

The overall purpose of these two lines is to analyze a log, extract the variants, and then sort them based on their frequencies in descending order. This can be useful in process mining to identify the most common or significant sequences of events in a given process.



In [ ]:
num_variants = len(variants)
num_variants
variants[:10]

###1.1.1.Variant Frequency Analyzing


The variants are sorted based on the frequency, and then a bar chart is drawn to visualize the frequency of the first 500 variants.

In [ ]:
values = [item[1] for item in variants[:500]]

plt.figure(figsize=(12, 8))
plt.bar(range(len(values)), values, color='skyblue', edgecolor='gray')

plt.title('Frequency Distribution of Top 500 Variants', fontsize=16)
plt.xlabel('Index of Variant', fontsize=12)
plt.ylabel('Variant\'s Frequency', fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.7)


plt.tight_layout()
plt.show()

This code calculates the cumulative percentage of frequencies of variants in the log file and plots it to visualize how many variants are needed to obtain 90% of the information stored in the log file.


This code defines a function called plot_accumulated_frequency that takes data as input. Our data contains a variant and its frequency. The function then calculates the accumulated frequency percentages and plots them against the variant index. It calculates the accumulated frequencies and total frequency of the variants in the data. Then, it calculates the accumulated frequency percentages by dividing each accumulated frequency by the total frequency and multiplying by 100. After lots of trial and error, it finds where 93% of the data is accumulated. The function proceeds to plot the accumulated frequency percentages against the variant index using matplotlib, and annotates the point where 93% of the data is accumulated on the plot.

In [ ]:
def plot_accumulated_frequency(data):

    frequencies = [item[1] for item in data]
    accumulated_frequencies = [sum(frequencies[:i+1]) for i in range(len(frequencies))]
    total_frequency = sum(frequencies)

    accumulated_frequency_percentages = [freq / total_frequency * 100 for freq in accumulated_frequencies]

    index_93_percent = next(idx for idx, val in enumerate(accumulated_frequency_percentages) if val >= 93)

    plt.figure(figsize=(10, 6))

    plt.plot(range(1, len(data)+1), accumulated_frequency_percentages, marker='o', color='pink', linestyle='-')

    plt.annotate(f'93% of data (Variant {index_93_percent + 1})',
                 xy=(index_93_percent + 1, accumulated_frequency_percentages[index_93_percent]),
                 xytext=(index_93_percent + 5, accumulated_frequency_percentages[index_93_percent] - 10),
                 arrowprops=dict(facecolor='black', arrowstyle='->', connectionstyle="arc3"),
                 fontsize=10, color='black')

    plt.xlabel('Variant Index', fontsize=12)
    plt.ylabel('Accumulated Frequency Percentage', fontsize=12)
    plt.title('Accumulated Frequency Percentage vs Variant Index', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)

    plt.tight_layout()
    plt.show()


plot_accumulated_frequency(variants)


###1.1.2.Activity Frequency Analyzing

In [ ]:
frequency = log["concept:name"].value_counts().reset_index()
frequency.columns = ["concept:name", "Frequency"]
total = frequency["Frequency"].sum()
frequency["Percentage"] = frequency["Frequency"] / total * 100
frequency.index = frequency.index + 1
frequency[:10]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def generate_heatmap_from_frequency(frequency_data, title='Heatmap'):

    df = pd.DataFrame(list(frequency_data.items()), columns=['Label', 'Frequency'])

    df = df.sort_values(by='Frequency', ascending=False)

    # Plot the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(df.set_index('Label'), cmap='YlGnBu', annot=True, fmt='d', linewidths=0.5)
    plt.title(title, fontsize=16)
    plt.xlabel('Label', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


keyword_data = frequency.set_index('concept:name')['Frequency'].to_dict()
generate_heatmap_from_frequency(keyword_data, title='Heatmap from Frequency Data')


In [ ]:
def plot_activity_frequency(df, n=10):
    frequency = df["concept:name"].value_counts().reset_index()
    frequency.columns = ["concept:name", "Frequency"]
    total = frequency["Frequency"].sum()
    frequency["Percentage"] = frequency["Frequency"] / total * 100
    frequency.index = frequency.index + 1

    top_activities = frequency[:n]
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(top_activities["concept:name"], top_activities["Percentage"], color='skyblue')
    plt.xlabel('Activity')
    plt.ylabel('Percentage')
    plt.title(f'Top {n} Most Frequent Activities')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


plot_activity_frequency(log, n=20)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import pm4py
from pm4py.objects.conversion.log import converter as log_converter

def plot_activity_frequency(df, n=10, save_log=False, log_filename="filtered_log.xes", file_dir='/content/drive/My Drive/BIS'):
    frequency = df["concept:name"].value_counts().reset_index()
    frequency.columns = ["concept:name", "Frequency"]
    total = frequency["Frequency"].sum()
    frequency["Percentage"] = frequency["Frequency"] / total * 100
    frequency.index = frequency.index + 1
    top_activities = frequency[:n]

    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(top_activities["concept:name"], top_activities["Percentage"], color='skyblue')
    plt.xlabel('Activity')
    plt.ylabel('Percentage')
    plt.title(f'Top {n} Most Frequent Activities')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Save filtered log if required
#     if save_log:
#         filtered_log = df[df["concept:name"].isin(top_activities["concept:name"])]
#         xes_log = log_converter.apply(filtered_log)
#         save_path = os.path.join(file_dir, log_filename)
#         pm4py.write_xes(xes_log, save_path)
# plot_activity_frequency(log, n=10, save_log=True, log_filename="filtered_log.xes", file_dir='/content/drive/My Drive/BIS')

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

keyword_data = dict(zip(frequency['concept:name'], frequency['Frequency']))

wordcloud = WordCloud(background_color='white', width=800, height=800)
wordcloud.generate_from_frequencies(keyword_data)

plt.figure(figsize=(7, 7))
plt.imshow(wordcloud, interpolation='bicubic')

plt.axis('off')
plt.show()


###1.1.4.Start and End Analyzing

In [ ]:
log_start = pm4py.get_start_activities(log)
log_end = pm4py.get_end_activities(log)

In [ ]:
log_start_df = pd.DataFrame.from_dict(log_start, orient='index', columns=['Occurrences'])
log_start_df = log_start_df.sort_values(by='Occurrences', ascending=False)

print(log_start_df)

In [ ]:
log_end_df = pd.DataFrame.from_dict(log_end, orient='index', columns=['Occurrences'])
log_end_df = log_end_df.sort_values(by='Occurrences', ascending=False)

print(log_end_df)

###1.1.5.Category Frequency Analyzing


When analyzing a dataset that includes categorical data, it's important to understand the frequency distribution of each category. This helps in identifying patterns, trends, and anomalies within the dataset. In this case, we're interested in the frequency of four specific categories: "3-way matching, invoice after goods receipt", "3-way matching, invoice before goods receipt", "2-way matching (no goods receipt needed)", and "Consignment".

The provided code calculates the frequency of each category in the dataset and visualizes it using a bar chart. It also adds percentage labels on top of each bar to provide a clear representation of the distribution.


The following Python code segment retrieves the list of start and end activities from the process log:


In [ ]:
import matplotlib.pyplot as plt

def plot_category_frequency(df):

    colors = ['blue', 'green', 'orange', 'red']
    category_frequencies = (df.groupby('case:Item Category').size() / len(df)) * 100

    plt.figure(figsize=(13, 6))
    plt.bar(category_frequencies.index, category_frequencies.values, color=colors)

    plt.xlabel('Item Category')
    plt.ylabel('Frequency')
    plt.title('Category Frequency')

    for i, v in enumerate(category_frequencies.values):
        plt.text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom')
    plt.show()


plot_category_frequency(log)


In [ ]:
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.conversion.log import converter as log_converter

In [ ]:
def export_categories(df, file_dir):

    categories = df['case:Item Category'].unique()

    for category in categories:
        category_df = df[df['case:Item Category'] == category]
        file_path = f"{file_dir}/{category}.xes"
        xes_log = log_converter.apply(category_df)
        pm4py.write_xes(xes_log, file_path)

# Assuming 'log' is your DataFrame
# file_dir = '/content/drive/My Drive/BIS'  # Directory where XES files will be saved
# export_categories(log, file_dir)

##1.2.Filtering and Processing Log File

#### Case Performance

In [ ]:
filtered_log = pm4py.filter_case_performance(log, 600, 31536000)

In [ ]:
print(filtered_log[:5])

In [ ]:
import matplotlib.pyplot as plt
import pm4py

num_cases_before_filtering = len(log)
filtered_log = pm4py.filter_case_performance(log, 600,31536000)
num_cases_after_filtering = len(filtered_log)
durations = pm4py.get_all_case_durations(filtered_log)

x_values = range(len(durations))
plt.figure(figsize=(8, 4))
plt.plot(x_values, durations)
middle_x = len(durations) // 2
middle_y = max(durations) / 2
plt.text(middle_x, middle_y, f'Number of cases after filtering: {len(durations)}',
  ha='center', va='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.5))

plt.show()

print(num_cases_before_filtering-num_cases_after_filtering)


Explanation for Notebook:
After the initial filter based on case duration, which removed 6,692 cases, we apply additional filters to refine our dataset further.



#### Top Variant Filter
The second filter we use is the "top-k variant" filter. We have identified that the top 599 variants account for 93% of the frequency in our dataset.
By applying this filter, we reduce the number of cases from 245,719 to 229706, eliminating 15,336 cases that fall outside the top 599 variants.
This helps in focusing on the most common variants, which likely represent the most significant and meaningful process paths.

In [ ]:
k = 599
filtered_log = pm4py.filter_variants_top_k(filtered_log, k)
print("Remaining cases after top-k variant filter:", len(pm4py.get_all_case_durations(filtered_log)))

# Output: Remaining cases after top-k variant filter: 229706
#15,336 removed


#### Start and End Activity Filter

The third filter focuses on the start activities of the cases. We only retain records that start with specific activities based on our understanding of activity frequency.
The activities we consider are:
"Create Purchase Order Item"
"Create Purchase Requisition Item"
"Vendor creates invoice"
"SRM: Created"
"Vendor creates debit memo"
This filter ensures that we only include cases that begin with these key activities, further refining our dataset to include the most relevant cases.

1.Filtering by Start Activities:

The provided code filters a process log based on specified start and end activities, and then prints the number of remaining cases in the filtered log. Here's a detailed breakdown of the code:


In [ ]:
filtered_log = pm4py.filter_start_activities(filtered_log, [
    "Create Purchase Order Item",
    "Create Purchase Requisition Item",
    "Vendor creates invoice",
    "SRM: Created",
    "Vendor creates debit memo"
])
num_cases_after_filtering = len(pm4py.get_all_case_durations(filtered_log))
print(num_cases_after_filtering)

 2.Filtering by End Activities:

This line filters the filtered_log to include only those cases that end with one of the specified activities. The activities considered are:
"Clear Invoice"
"Record Invoice Receipt"
"Record Goods Receipt"
"Delete Purchase Order Item"
"Remove Payment Block"
"Cancel Invoice Receipt"
"Change Approval for Purchase Order"
"Change Delivery Indicator"
"Cancel Goods Receipt"
"Receive Order Confirmation"
"Block Purchase Order Item"
"Set Payment Block"
"SRM: Change was Transmitted"
"SRM: Transfer Failed (E.Sys.)"
"SRM: In Transfer to Execution Syst."
"Cancel Subsequent Invoice"
"SRM: Deleted"
"SRM: Transaction Completed"
"Update Order Confirmation"


In [ ]:
filtered_log = pm4py.filter_end_activities(filtered_log, [
    "Clear Invoice",
    "Record Invoice Receipt",
    "Record Goods Receipt",
    "Delete Purchase Order Item",
    "Remove Payment Block",
    "Cancel Invoice Receipt",
    "Change Approval for Purchase Order",
    "Change Delivery Indicator",
    "Cancel Goods Receipt",
    "Receive Order Confirmation",
    "Block Purchase Order Item",
    "Set Payment Block",
    "SRM: Change was Transmitted",
    "SRM: Transfer Failed (E.Sys.)",
    "SRM: In Transfer to Execution Syst.",
    "Cancel Subsequent Invoice",
    "SRM: Deleted",
    "SRM: Transaction Completed",
    "Update Order Confirmation"
])
num_cases_after_filtering = len(pm4py.get_all_case_durations(filtered_log))
print(num_cases_after_filtering)

In [ ]:
print("Remaining cases:", len(pm4py.get_all_case_durations(filtered_log)))

#### Activity Rework

In [ ]:
cases_to_exclude = pd.concat([
    pm4py.filter_activities_rework(filtered_log, "Clear Invoice", 2),
    pm4py.filter_activities_rework(filtered_log, "Record Goods Receipt", 2),
    pm4py.filter_activities_rework(filtered_log, "Record Invoice Receipt", 2)
    ])
cases_to_exclude = cases_to_exclude["case:concept:name"].unique()

print("cases to exclude:",len(cases_to_exclude))
print("total cases before filtering:", len(filtered_log["case:concept:name"].unique()))
filtered_log = filtered_log[~filtered_log['case:concept:name'].isin(cases_to_exclude)]
print("total cases after filtering:", len(filtered_log["case:concept:name"].unique()))

#2.Process Discovery

##2.1.Alpha Miner

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_alpha(log)
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_alpha(filtered_log)
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
# fitness for alpha miner:
print("replay based fitness:",pm4py.fitness_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("replay based precision:",pm4py.precision_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("generalization:",generalization_evaluator.apply(filtered_log, net, initial_marking, final_marking))
print("simplicity:",simplicity_evaluator.apply(net))

##2.2.Inductive Miner

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(log)
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(filtered_log)
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
print("replay based fitness:",pm4py.fitness_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("replay based precision:",pm4py.precision_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("generalization:",generalization_evaluator.apply(filtered_log, net, initial_marking, final_marking))
print("simplicity:",simplicity_evaluator.apply(net))

##2.3.Heuristic Miner

In [ ]:
heu_net = pm4py.discover_heuristics_net(log, dependency_threshold=0.99,and_threshold = 0.99,loop_two_threshold=0.99)
pm4py.view_heuristics_net(heu_net)


In [ ]:
heu_net = pm4py.discover_heuristics_net(filtered_log, dependency_threshold=0.99,and_threshold = 0.99,loop_two_threshold=0.99)
pm4py.view_heuristics_net(heu_net)


In [ ]:
#Petri-net of huristic miner:
net, im, fm = pm4py.discover_petri_net_heuristics(log, dependency_threshold=0.99,and_threshold = 0.99,loop_two_threshold=0.99)
pm4py.view_petri_net(net, im, fm)

In [ ]:
net, im, fm = pm4py.discover_petri_net_heuristics(filtered_log, dependency_threshold=0.99,and_threshold = 0.99,loop_two_threshold=0.99)
pm4py.view_petri_net(net, im, fm)

In [ ]:
# fitness for huristic miner:
print("replay based fitness:",pm4py.fitness_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("replay based precision:",pm4py.precision_token_based_replay(filtered_log, net, initial_marking, final_marking))
print("generalization:",generalization_evaluator.apply(filtered_log, net, initial_marking, final_marking))
print("simplicity:",simplicity_evaluator.apply(net))

##2.4.Process Tree

In [ ]:
tree = pm4py.discover_process_tree_inductive(log)
pm4py.view_process_tree(tree)

In [ ]:
tree = pm4py.discover_process_tree_inductive(filtered_log)
pm4py.view_process_tree(tree)

##2.5.Performance Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Data after filtering
data_filtered = {
    'Algorithm': ['Alpha Miner', 'Inductive Miner', 'Heuristic Miner'],
    'Fitness_filtered': [0.52, 1.0, 0.72],
    'Precision_filtered': [0.49, 0.27, 1.0],
    'Generalization_filtered': [0.96, 0.97, 0.50],
    'Simplicity_filtered': [0.34, 0.63, 0.50]
}

# Data before filtering
data_unfiltered = {
    'Algorithm': ['Alpha Miner', 'Inductive Miner', 'Heuristic Miner'],
    'Fitness_unfiltered': [0.16, 1.0, 0.73],
    'Precision_unfiltered': [0.16, 1.0, 0.73],
    'Generalization_unfiltered': [0.97, 0.97, 0.60],
    'Simplicity_unfiltered': [0.46, 0.63, 0.50]
}

df_filtered = pd.DataFrame(data_filtered)
df_unfiltered = pd.DataFrame(data_unfiltered)

df_combined = pd.merge(df_filtered, df_unfiltered, on='Algorithm')
df_combined.set_index('Algorithm', inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

algorithms = df_combined.index
colors = ['b', 'r']
for ax, algorithm in zip(axes, algorithms):
    filtered_values = df_combined.loc[algorithm, ['Fitness_filtered', 'Precision_filtered', 'Generalization_filtered', 'Simplicity_filtered']]
    unfiltered_values = df_combined.loc[algorithm, ['Fitness_unfiltered', 'Precision_unfiltered', 'Generalization_unfiltered', 'Simplicity_unfiltered']]

    index = np.arange(len(filtered_values))
    bar_width = 0.35

    ax.bar(index, unfiltered_values, bar_width, label='Before Filtering', color=colors[0])
    ax.bar(index + bar_width, filtered_values, bar_width, label='After Filtering', color=colors[1])

    ax.set_title(f'{algorithm}')
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Values')
    ax.set_xticks(index + bar_width / 2)
    ax.set_xticklabels(['Fitness', 'Precision', 'Generalization', 'Simplicity'])
    ax.legend()

plt.suptitle('Performance Metrics of Process Mining Algorithms')
plt.show()


#3.Data Segmentation


In [ ]:
def categorizeLogFile(log,categories):
  return {
        item: pm4py.filter_event_attribute_values(log, "case:Item Category", [item], level="case", retain=True)
        for item in categories
    }
segmentedData = categorizeLogFile(log,["2-way match", "3-way match, invoice after GR","3-way match, invoice before GR", "Consignment"])

##3.1.Filtering

In [ ]:
#apply filters:
def filter(input):
  log = input.copy()
  log = pm4py.filter_case_performance(log,  600, 31536000)
  log = pm4py.filter_variants_top_k(log, k)
  log = pm4py.filter_start_activities(log,["Create Purchase Order Item","Create Purchase Requisition Item","Vendor creates invoice","SRM: Created","Vendor creates debit memo"])
  log = pm4py.filter_end_activities(log, [
  "Clear Invoice",
  "Record Invoice Receipt",
  "Record Goods Receipt",
  "Delete Purchase Order Item",
  "Remove Payment Block",
  "Cancel Invoice Receipt",
  "Change Approval for Purchase Order",
  "Change Delivery Indicator",
  "Cancel Goods Receipt",
  "Receive Order Confirmation",
  "Block Purchase Order Item",
  "Set Payment Block",
  "SRM: Change was Transmitted",
  "SRM: Transfer Failed (E.Sys.)",
  "SRM: In Transfer to Execution Syst.",
  "Cancel Subsequent Invoice",
  "SRM: Deleted",
  "SRM: Transaction Completed",
  "Update Order Confirmation"])
  cases_to_exclude = pd.concat([
    pm4py.filter_activities_rework(log, "Clear Invoice", 2),
    pm4py.filter_activities_rework(log, "Record Goods Receipt", 2),
    pm4py.filter_activities_rework(log, "Record Invoice Receipt", 2)
    ])
  cases_to_exclude = cases_to_exclude["case:concept:name"].unique()
  log = log[~log['case:concept:name'].isin(cases_to_exclude)]
  return log

In [ ]:
for item in segmentedData:
  print("number of cases in",item," before filtering:", len(segmentedData[item]["case:concept:name"].unique()))
  segmentedData[item] = filter(segmentedData[item])
  print("number of cases in",item," after filtering:", len(segmentedData[item]["case:concept:name"].unique()))
  print("\n")

In [ ]:
#for the 3-way match, invoice after GR, it should contains "Record Goods Receipt" and also "Record Invoice Receipt"
segmentedData["3-way match, invoice after GR"] = pm4py.filter_activities_rework(segmentedData["3-way match, invoice after GR"], "Record Invoice Receipt", 1)
segmentedData["3-way match, invoice after GR"] = pm4py.filter_activities_rework(segmentedData["3-way match, invoice after GR"], "Record Goods Receipt", 1)
print("size of \"3-way match, invoice after GR\" after filter:",len(segmentedData["3-way match, invoice after GR"]["case:concept:name"].unique()))

In [ ]:
#for the 3-way matching, invoice before goods receipt, it should contains "Record Goods Receipt"
segmentedData["3-way match, invoice before GR"] = pm4py.filter_activities_rework(segmentedData["3-way match, invoice before GR"], "Record Goods Receipt", 1)
print("size of \"3-way match, invoice before GR\" after filter:",len(segmentedData["3-way match, invoice before GR"]["case:concept:name"].unique()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = [
    '2-way match',
    '3-way match, invoice after GR',
    '3-way match, invoice before GR',
    'Consignment'
]
before_filtering = [677, 9252, 221010, 14498]
after_filtering = [651, 8253, 189687, 12435]

x = np.arange(len(categories))

width = 0.35
fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, before_filtering, width, label='Before Filtering', color='skyblue')
bars2 = ax.bar(x + width/2, after_filtering, width, label='After Filtering', color='steelblue')

for bar in bars1:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval, int(yval), va='bottom', ha='center')

for bar in bars2:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval, int(yval), va='bottom', ha='center')

ax.set_xlabel('Categories')
ax.set_ylabel('Number of Cases')
ax.set_title('Cases Comparison')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

plt.show()


In [ ]:
for item in segmentedData:
  print(item)

##3.2.Process Discovery

In [ ]:
#alpha miner
alpha_miner_heatmap = {}
for item in segmentedData:
  net, initial_marking, final_marking = pm4py.discover_petri_net_alpha(segmentedData[item])
  pm4py.view_petri_net(net, initial_marking, final_marking)

  fitness = pm4py.fitness_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  precision = pm4py.precision_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  generalization = generalization_evaluator.apply(segmentedData[item], net, initial_marking, final_marking)
  simplicity = simplicity_evaluator.apply(net)

  alpha_miner_heatmap[item] = [fitness["log_fitness"],precision,generalization,simplicity]

  print("replay based fitness:", fitness)
  print("replay based precision:", precision)
  print("generalization:",generalization)
  print("simplicity:",simplicity)

In [ ]:
# inductive minder:
inductive_miner_heatmap = {}
for item in segmentedData:
  net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(segmentedData[item])
  pm4py.view_petri_net(net, initial_marking, final_marking)

  fitness = pm4py.fitness_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  precision = pm4py.precision_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  generalization = generalization_evaluator.apply(segmentedData[item], net, initial_marking, final_marking)
  simplicity = simplicity_evaluator.apply(net)

  inductive_miner_heatmap[item] = [fitness["log_fitness"],precision,generalization,simplicity]

  print("replay based fitness:", fitness)
  print("replay based precision:", precision)
  print("generalization:",generalization)
  print("simplicity:",simplicity)

In [ ]:
#Heuristic miner:
heuristic_miner_heatmap = {}
for item in segmentedData:
  net, initial_marking, final_marking = pm4py.discover_petri_net_heuristics(segmentedData[item], dependency_threshold=0.99,and_threshold = 0.99,loop_two_threshold=0.99)
  pm4py.view_petri_net(net, initial_marking, final_marking)

  fitness = pm4py.fitness_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  precision = pm4py.precision_token_based_replay(segmentedData[item], net, initial_marking, final_marking)
  generalization = generalization_evaluator.apply(segmentedData[item], net, initial_marking, final_marking)
  simplicity = simplicity_evaluator.apply(net)

  heuristic_miner_heatmap[item] = [fitness["log_fitness"],precision,generalization,simplicity]

  print("replay based fitness:", fitness)
  print("replay based precision:", precision)
  print("generalization:",generalization)
  print("simplicity:",simplicity)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

performance_metrics = {
    '2-way match': {
        'alpha miner': [0.54, 0.68, 0.68, 0.5],
        'inductive miner': [1, 0.35, 0.74, 0.64],
        'heuristic miner': [0.83, 0.60, 0.83, 0.86]
    },
    '3-way match, invoice after GR': {
        'alpha miner': [0.48, 0, 0.86, 0.39],
        'inductive miner': [1 ,0.32,0.90,0.63],
        'heuristic miner': [0.93, 0.73, 0.95, 0.70]
    },
    '3-way match, invoice before GR': {
        'alpha miner':  [0.51, 0, 0.92, 0.33],
        'inductive miner': [1 , 0.21,0.96, 0.62],
        'heuristic miner':[0.95, 0.63,0.95, 0.60]
    },
    'Consignment': {
        'alpha miner':  [0.51, 0.20, 0.74, 0.44],
        'inductive miner': [1,0.84 , 0.83,0.63],
        'heuristic miner':[0.99, 0.99,0.94, 0.66]
    }
}

categories = list(performance_metrics.keys())


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(25, 12))
colors = ['#FFC0CB', '#ADD8E6', '#90EE90']

for i, category in enumerate(categories):
    row = i // 2
    col = i % 2
    ax = axes[row, col]

    alpha_values = performance_metrics[category]["alpha miner"]
    inductive_values = performance_metrics[category]["inductive miner"]
    heuristic_values = performance_metrics[category]["heuristic miner"]

    bar_width = 0.2
    index = np.arange(len(alpha_values))

    ax.bar(index, alpha_values, bar_width, label='Alpha Miner', color=colors[0])
    ax.bar(index + bar_width, inductive_values, bar_width, label='Inductive Miner', color=colors[1])
    ax.bar(index + 2 * bar_width, heuristic_values, bar_width, label='Heuristic Miner', color=colors[2])

    ax.set_xlabel('Metrics')
    ax.set_ylabel('Values')
    ax.set_title(category)
    ax.set_xticks(index + bar_width)
    ax.set_xticklabels(['Fitness', 'Precision', 'Generalization', 'Simplicity'][:len(alpha_values)])  # Adjust based on the number of metrics
    ax.legend()

plt.tight_layout()
plt.show()
